<a href="https://colab.research.google.com/github/LS-SEC/ls-report-template/blob/main/%EC%9D%80%ED%96%89_%EC%9D%BC%EC%9D%BC%EB%8D%B0%EC%9D%B4%ED%84%B0_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q --upgrade pykrx tabulate yfinance

In [4]:
import os
from importlib.metadata import version
from google.colab import userdata

os.environ['KRX_ID'] = userdata.get('KRX_ID')   # 금고에서 아이디 꺼내기
os.environ['KRX_PW'] = userdata.get('KRX_PW')   # 금고에서 비번 꺼내기

from pykrx import stock   # ← 반드시 자격증명 설정 '후에' import

_t = stock.get_index_ohlcv("20240102", "20240105", "1001")
print(f"pykrx {version('pykrx')}")
print("✓ KRX 로그인 정상" if not _t.empty else "⚠ 로그인 실패 — Secret 확인")


KRX 로그인 시도...
  로그인 ID: mujige01
KRX 로그인 완료.
  로그인 시간: 2026-07-06 09:35:20
  만료 시간: 2026-07-06 10:35:20
pykrx 1.2.8
✓ KRX 로그인 정상


In [5]:
STOCKS = {
    '105560': 'KB금융', '055550': '신한지주', '086790': '하나금융지주',
    '316140': '우리금융지주', '024110': '기업은행', '138930': 'BNK금융지주',
    '139130': 'iM금융지주', '175330': 'JB금융지주', '323410': '카카오뱅크',
}

In [6]:
# ================================================================
# [셀 4 · v2] 데이터 추출 — 시계열 + 수급심화 + 밸류에이션 + 대외지표
#
#  전제: 셀1(설치)·셀2(로그인)·셀3(STOCKS) 실행 후 이 셀 실행
#  셀1은 아래로 교체:  !pip install -q --upgrade pykrx tabulate yfinance
#
#  ⚠️ 첫 실행 시 확인할 [CHECK] 3곳 — 오류 나면 해당 주석의 대안으로 조정
#   [CHECK A] 밸류에이션 함수 시그니처
#   [CHECK B] 종목 단위 투자자 수급 조회
#   [CHECK C] yfinance 대외지표 (미10y는 /10 보정)
# ================================================================
import datetime as _dt
from zoneinfo import ZoneInfo
from html import escape
import pandas as pd
from IPython.display import HTML, display

KST = ZoneInfo('Asia/Seoul')
INDICES = {'1001': '코스피', '2001': '코스닥'}
SECTORS = {'5044':'반도체','5043':'자동차','5045':'헬스케어','5046':'은행',
           '5048':'에너지화학','5049':'철강','5051':'방송통신','5052':'건설',
           '5054':'증권','5055':'기계장비','5056':'보험','5057':'운송',
           '5061':'경기소비재','5062':'필수소비재','5063':'K콘텐츠',
           '5064':'정보기술','5065':'유틸리티'}

# ---------- 1) 기준일 (16시 이후=당일 확정치, 이전=직전 영업일) ----------
_now = _dt.datetime.now(KST)
_e = _now.strftime('%Y%m%d')
_probe = [d.strftime('%Y%m%d') for d in stock.get_index_ohlcv((_now - _dt.timedelta(days=14)).strftime('%Y%m%d'), _e, '1001').index]
if _now.hour < 16 and _probe and _probe[-1] == _e:
    _probe = _probe[:-1]
BASE = _probe[-1]
BASE_F = f'{BASE[:4]}-{BASE[4:6]}-{BASE[6:]}'
YEAR = int(BASE[:4])
_FROM = f'{YEAR-1}1201'          # 전년 12월부터 조회 → YTD 기준(전년 마지막 종가) 확보
print('기준일:', BASE_F)

# ---------- 2) 수익률 계산기 (1D/1W/1M/YTD) ----------
def _rets(close, d1_override=None):
    """종가 시리즈 → (종가, 1D, 1W, 1M, YTD) / 데이터 부족 항목은 None"""
    last = close.iloc[-1]
    def pct(n):
        return (last / close.iloc[-1 - n] - 1) * 100 if len(close) > n else None
    d1 = d1_override if d1_override is not None else pct(1)
    w1, m1 = pct(5), pct(21)                    # 1W=5영업일, 1M=21영업일
    prev = close[close.index.year < YEAR]
    ytd = (last / prev.iloc[-1] - 1) * 100 if len(prev) else None
    return last, d1, w1, m1, ytd

def _f(x, nd=2, comma=False):
    if x is None: return 'N/A'
    return f'{x:,.{nd}f}' if comma else f'{x:.{nd}f}'

# ---------- 3) 지수 · 업종 (시계열) ----------
print('지수·업종 수집 (연초부터, 30~60초)...')
def _idx_row(code, name):
    c = stock.get_index_ohlcv(_FROM, BASE, code)['종가']
    last, d1, w1, m1, ytd = _rets(c)
    return [name, _f(last, 2, True), _f(d1), _f(w1), _f(m1), _f(ytd)]

idx_rows = [_idx_row(c, n) for c, n in INDICES.items()]
sec_rows = [_idx_row(c, n) for c, n in SECTORS.items()]
sec_rows.sort(key=lambda r: float(r[2]), reverse=True)   # 1D 내림차순
COLS_TS = ['구분','종가','1D(%)','1W(%)','1M(%)','YTD(%)']
idx_md = pd.DataFrame(idx_rows, columns=COLS_TS).to_markdown(index=False)
sec_md = pd.DataFrame(sec_rows, columns=COLS_TS).to_markdown(index=False)

# ---------- 4) 수급 (시장) + 수급상세 (기관 분해 — 서술용) ----------
print('수급 수집...')
flow_rows, detail_rows = [], []
for mkt, name in [('KOSPI','코스피'), ('KOSDAQ','코스닥')]:
    inv = stock.get_market_trading_value_by_investor(BASE, BASE, mkt)['순매수']
    ind = inv['개인']/1e9
    frn = (inv['외국인'] + inv['기타외국인'])/1e9
    ins = inv['기관합계']/1e9
    flow_rows.append([name, f'{ind:+,.1f}', f'{frn:+,.1f}', f'{ins:+,.1f}'])
    if mkt == 'KOSPI':   # 기관 세부는 코스피만 (해석 서술용, 보고서 표 미반영)
        for k in ['금융투자','투신','보험','연기금']:
            if k in inv.index:
                detail_rows.append([k, f'{inv[k]/1e9:+,.1f}'])
flow_md = pd.DataFrame(flow_rows, columns=['시장','개인(십억)','외국인(십억)','기관합계(십억)']).to_markdown(index=False)
detail_md = pd.DataFrame(detail_rows, columns=['주체(코스피)','순매수(십억)']).to_markdown(index=False)

# ---------- 5) 관심종목: 가격·수익률 + 수급 + 밸류 ----------
print('관심종목 수집...')
# [CHECK A] 밸류에이션 — 오류 시 대안: stock.get_market_fundamental_by_ticker(BASE, market="ALL")
try:
    _fund = stock.get_market_fundamental(BASE, market='ALL')
except Exception:
    _fund = stock.get_market_fundamental_by_ticker(BASE, market='ALL')

def _stk_flow(code):
    # [CHECK B] 종목 단위 수급 — 오류 시 이 블록 전체를 None 반환으로 두고 진행 (표에 N/A)
    try:
        inv = stock.get_market_trading_value_by_investor(BASE, BASE, code)['순매수']
        frn = (inv.get('외국인', 0) + inv.get('기타외국인', 0)) / 1e8   # 억원
        ins = inv.get('기관합계', 0) / 1e8
        return frn, ins
    except Exception:
        return None, None

price_rows, val_rows = [], []
for code, name in STOCKS.items():
    df = stock.get_market_ohlcv(_FROM, BASE, code)
    d1_official = df['등락률'].iloc[-1] if '등락률' in df.columns else None
    last, d1, w1, m1, ytd = _rets(df['종가'], d1_override=d1_official)
    price_rows.append([name, f'{int(last):,}', _f(d1), _f(w1), _f(m1), _f(ytd)])
    frn, ins = _stk_flow(code)
    per = _fund.loc[code, 'PER'] if code in _fund.index else None
    pbr = _fund.loc[code, 'PBR'] if code in _fund.index else None
    dvd = _fund.loc[code, 'DIV'] if code in _fund.index else None
    val_rows.append([name,
                     f'{frn:+,.0f}' if frn is not None else 'N/A',
                     f'{ins:+,.0f}' if ins is not None else 'N/A',
                     _f(per), _f(pbr), _f(dvd)])
price_rows.sort(key=lambda r: float(r[2]) if r[2] != 'N/A' else -999, reverse=True)  # 1D 내림차순
_order = [r[0] for r in price_rows]
val_rows.sort(key=lambda r: _order.index(r[0]))                                      # 표B는 표A 순서 동일
stk_md = pd.DataFrame(price_rows, columns=['종목','종가','1D(%)','1W(%)','1M(%)','YTD(%)']).to_markdown(index=False)
val_md = pd.DataFrame(val_rows, columns=['종목','외국인(억)','기관(억)','PER(배)','PBR(배)','배당수익률(%)']).to_markdown(index=False)

# ---------- 6) 대외지표 (yfinance) ----------
print('대외지표 수집...')
ext_rows, ext_date = [], ''
try:
    import yfinance as yf
    EXT = [('^GSPC','S&P500'), ('^IXIC','나스닥'), ('^SOX','필라델피아반도체'),
           ('KRW=X','달러/원'), ('CL=F','WTI'), ('^TNX','美 10년물')]
    for tk, name in EXT:
        try:
            h = yf.Ticker(tk).history(period='15d')['Close'].dropna()
            h = h[[d.strftime('%Y%m%d') < BASE for d in h.index]]   # 기준일 '간밤'까지의 세션만
            last, prev = float(h.iloc[-1]), float(h.iloc[-2])
            if tk == '^TNX':   # [CHECK C] ^TNX는 수익률×10으로 제공 → /10 보정, 등락은 %p
                last, prev = last/10, prev/10
                ext_rows.append([name, f'{last:.2f}', f'{last-prev:+.2f}%p'])
            else:
                ext_rows.append([name, f'{last:,.2f}', f'{(last/prev-1)*100:+.2f}'])
            ext_date = h.index[-1].strftime('%m/%d')
        except Exception:
            ext_rows.append([name, 'N/A', 'N/A'])
except Exception as e:
    print('yfinance 실패(대외 표 생략 가능):', e)
ext_md = pd.DataFrame(ext_rows, columns=['구분','종가','등락률(%)']).to_markdown(index=False) if ext_rows else '(수집 실패)'

# ---------- 7) 조립 + [전체 복사] ----------
report = (f"## 대외 (미국 {ext_date} 마감 · 美10년물 등락은 %p)\n{ext_md}\n\n"
          f"## 지수 (기준일 {BASE_F})\n{idx_md}\n\n"
          f"## 수급 (순매수, 십억원)\n{flow_md}\n\n"
          f"## 수급상세 (코스피 기관 분해 — 서술용)\n{detail_md}\n\n"
          f"## 업종 (KRX 섹터지수)\n{sec_md}\n\n"
          f"## 관심종목\n{stk_md}\n\n"
          f"## 관심종목 수급·밸류 (수급: 당일 순매수 억원 / PER·PBR·배당: 트레일링)\n{val_md}")
display(HTML(
    "<button style='padding:8px 18px;font-size:14px;margin:8px 0;cursor:pointer;' "
    "onclick=\"navigator.clipboard.writeText(document.getElementById('rpt').value)"
    ".then(()=>this.textContent='복사 완료!')\">[전체 복사]</button>"
    f"<textarea id='rpt' rows='30' style='width:100%;font-family:monospace;font-size:12px;'>{escape(report)}</textarea>"
))
print('버튼이 안 눌리면: 상자 클릭 → Ctrl+A → Ctrl+C')

기준일: 2026-07-06
지수·업종 수집 (연초부터, 30~60초)...
수급 수집...
관심종목 수집...
대외지표 수집...


버튼이 안 눌리면: 상자 클릭 → Ctrl+A → Ctrl+C
